# Diagnóstico de marca empleadora — Deloitte vs. PwC, EY y KPMG
### Sentimiento (transformers) + topic modeling (BERTopic) sobre reseñas de Glassdoor

**Encargo de RRHH:** identificar los puntos fuertes de Deloitte como empleador (para explotarlos en *employer branding* y selección) y las áreas de mejora que dañan su reputación o provocan salidas, **siempre comparado con sus competidores directos**: un problema solo es prioritario si estamos peor que el mercado.

**Empresa y peers.** Elijo Deloitte y comparo con PwC, EY y KPMG, las otras tres Big 4: compiten por el mismo perfil de graduado y ofrecen condiciones similares, por lo que sus reseñas son comparables.

| Sección | Contenido |
|---|---|
| 0 | Carga y preparación |
| 1 | EDA: volumen, rating, recomendación, actuales vs. antiguos, evolución |
| 2 | Sentimiento en pros y cons (control de calidad) |
| 3 | Topic modeling con BERTopic y categorías de negocio |
| 4 | Comparación con peers: peso, balance neto y contraste con el rating |
| 5 | Empleados actuales vs. antiguos (motivos de salida) |
| 6 | Matriz de priorización, recomendaciones y limitaciones |
| 7 | Extras: antes/después de 2020, antigüedad y recomendación |
| — | Conclusiones de negocio |

**Reproducibilidad.** Semilla fija (`SEED = 42`), muestra fija por empresa y caché en disco (`cache/`) de los pasos pesados (sentimiento, embeddings y topics). Todas las gráficas se exportan a `figuras/` (versión con título) y `figuras/slides/` (sin título, para la presentación).

In [ ]:
# Instalación (solo necesaria en Google Colab o si faltan librerías).
# transformers y torch ya vienen en Colab; BERTopic y sentence-transformers no.
import importlib.util, subprocess, sys
faltan = [p for p, m in [('bertopic', 'bertopic'), ('sentence-transformers', 'sentence_transformers'),
                         ('pyarrow', 'pyarrow')] if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltan])
print('Dependencias OK' if not faltan else f'Instalado: {faltan}')

---
## 0. Carga y preparación

Configuración general. Decisiones clave:
- **Ventana temporal**: julio 2018 – julio 2023 (la que fija el enunciado). Se comprueba igualmente sobre los datos.
- **Muestra**: máximo 2.000 reseñas por empresa con semilla fija. Con el mismo tamaño para las cuatro, ninguna domina el modelo de topics conjunto y la "media de los peers" pesa igual a cada competidor.
- **GPU**: si hay GPU se usa. Sin GPU el notebook funciona igual (≈15-20 min la primera vez en un portátil; después tira de caché). Si se quiere acelerar en CPU, `REDUCIR_SIN_GPU = True` baja la muestra a 1.000 por empresa.

In [ ]:
import os, re, json, time, random, textwrap, hashlib, math, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

# ---- Parámetros del análisis ----
TARGET = 'Deloitte'
PEERS = ['PwC', 'EY', 'KPMG']
FIRMS = [TARGET] + PEERS
# 'Ey' es una segunda página de EY en Glassdoor (procede del segundo dataset): se unifica con 'EY'
ALIAS = {'Ey': 'EY'}
FECHA_INI, FECHA_FIN = pd.Timestamp('2018-07-01'), pd.Timestamp('2023-07-31')
MIN_RESENAS = 300

SEED = 42
MAX_POR_EMPRESA = 2000
REDUCIR_SIN_GPU = False      # True -> 1.000 reseñas/empresa si no hay GPU
MAX_POR_EMPRESA_CPU = 1000

# ---- Rutas (en Colab, sube el parquet a /content o cambia DATA_PATH) ----
DATA_PATH = Path('glassdoor_reviews_hr.parquet')
CACHE_DIR, FIG_DIR = Path('cache'), Path('figuras')
for p in [CACHE_DIR, FIG_DIR, FIG_DIR / 'slides', Path('resultados')]:
    p.mkdir(parents=True, exist_ok=True)

# ---- Semillas y dispositivo ----
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DEVICE = 0 if GPU else -1
N_MUESTRA = MAX_POR_EMPRESA if (GPU or not REDUCIR_SIN_GPU) else MAX_POR_EMPRESA_CPU
print(f'GPU disponible: {GPU} | Reseñas máximas por empresa: {N_MUESTRA}')

# Diccionario donde se van guardando las cifras clave (alimenta la presentación)
RES = {}

**Estilo gráfico común.** Deloitte en verde (color de acento) y los peers en grises; los títulos de cada gráfico enuncian la conclusión y se calculan a partir de los datos, no se escriben a mano.

In [ ]:
VERDE = '#26890D'          # acento: Deloitte
GRIS_PEERS = '#8E99A6'     # media de peers
COLORES = {'Deloitte': VERDE, 'PwC': '#4A5562', 'EY': '#8E99A6', 'KPMG': '#BAC2CB'}
ROJO, ROJO_SUAVE, VERDE_SUAVE = '#C0392B', '#E8A39B', '#A9D18E'
TINTA, TINTA2 = '#1F2328', '#5F6368'

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'figure.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'axes.titlecolor': TINTA,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#BDC1C6',
    'axes.grid': True, 'grid.color': '#ECEEF0', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.labelcolor': TINTA2, 'axes.labelsize': 11.5, 'xtick.color': TINTA2, 'ytick.color': TINTA2,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.frameon': False, 'legend.fontsize': 11,
})

def figura(titulo, subtitulo=None, ncols=1, nrows=1, figsize=(12, 6), **kw):
    """Crea una figura con título-conclusión y subtítulo alineados a la izquierda (medidas en pulgadas
    para que la cabecera no se solape con los ejes, sea cual sea el tamaño de la figura)."""
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kw)
    W, H = figsize
    t = textwrap.fill(titulo, int(W * 6.8))
    s = textwrap.fill(subtitulo, int(W * 10.5)) if subtitulo else ''
    n_t, n_s = t.count('\n') + 1, (s.count('\n') + 1 if s else 0)
    fig._titulos = [fig.text(0.01, 1 - 0.08 / H, t, ha='left', va='top', fontsize=16, fontweight='bold', color=TINTA)]
    if s:
        fig._titulos.append(fig.text(0.01, 1 - (0.16 + n_t * 0.30) / H, s, ha='left', va='top', fontsize=11.5, color=TINTA2))
    fig._top = 1 - (0.30 + n_t * 0.30 + n_s * 0.22) / H
    return fig, axes

def guardar(fig, nombre):
    """Guarda la versión con título (notebook) y otra sin título (para las diapositivas)."""
    fig.tight_layout(rect=(0, 0, 1, fig._top))
    fig.savefig(FIG_DIR / f'{nombre}.png', bbox_inches='tight', facecolor='white')
    for t in fig._titulos:
        t.set_visible(False)
    fig.savefig(FIG_DIR / 'slides' / f'{nombre}.png', bbox_inches='tight', facecolor='white')
    for t in fig._titulos:
        t.set_visible(True)
    plt.show()

def separar(vals, min_dist):
    """Desplaza valores (posiciones de etiquetas) para que queden al menos a `min_dist`, sin cambiar su orden."""
    vals = np.asarray(vals, float); res = vals.copy(); orden = np.argsort(vals)
    for k in range(1, len(orden)):
        i, j = orden[k], orden[k - 1]
        res[i] = max(res[i], res[j] + min_dist)
    return res - (res.mean() - vals.mean())

def posicion(serie, mayor_mejor=True):
    """Posición de Deloitte dentro de las 4 empresas (1 = mejor)."""
    return int(serie.rank(ascending=not mayor_mejor, method='min')[TARGET])

def pct_recomienda(s):
    """% que recomienda sobre quienes opinan (Sí + No). 'Neutral' = sin opinión."""
    op = s.isin(['Sí', 'No'])
    return 100 * (s[op] == 'Sí').mean() if op.any() else np.nan

def etiquetar_barras(ax, fmt='{:.1f}', dentro=False, color=TINTA, size=11):
    for p in ax.patches:
        w = p.get_width()
        if np.isnan(w) or w == 0:
            continue
        ax.text(w + (0 if dentro else ax.get_xlim()[1] * 0.01), p.get_y() + p.get_height() / 2, fmt.format(w),
                va='center', ha='left', color=color, fontsize=size)

### 0.1 Carga filtrada
Solo se leen las filas de las cuatro empresas (y del alias `Ey`) con el filtro de `pd.read_parquet`, sin cargar el fichero entero (~5 M de reseñas).

In [ ]:
raw = pd.read_parquet(DATA_PATH, filters=[('firm', 'in', FIRMS + list(ALIAS))])
print('Reseñas por nombre de empresa en el fichero:')
print(raw['firm'].value_counts().to_string())

# ¿'Ey' duplica reseñas de 'EY'? Se cruzan por texto de pros y cons
solape = raw[raw.firm == 'Ey'].merge(raw[raw.firm == 'EY'], on=['pros', 'cons']).shape[0]
print(f"\n'Ey': {(raw.firm == 'Ey').sum():,} reseñas ({raw.loc[raw.firm == 'Ey', 'date_review'].min()} a "
      f"{raw.loc[raw.firm == 'Ey', 'date_review'].max()}); textos idénticos a 'EY': {solape}")
raw['firm'] = raw['firm'].replace(ALIAS)
raw['date_review'] = pd.to_datetime(raw['date_review'])
print(f"Rango de fechas en el fichero: {raw.date_review.min().date()} a {raw.date_review.max().date()}")

`Ey` es otra página de Glassdoor de la misma firma (cubre 2021-2023, procede del dataset global) y prácticamente no duplica textos con `EY`. **Decisión: se unifica con EY** y se eliminan los pocos textos duplicados. Separarla partiría a EY en dos "empresas" y la dejaría infrarrepresentada en los años recientes.

### 0.2 Ventana temporal, duplicados, idioma y limpieza de texto

In [ ]:
n0 = len(raw)
pob = raw[(raw.date_review >= FECHA_INI) & (raw.date_review <= FECHA_FIN)].copy()
print(f'Fuera de la ventana jul-2018/jul-2023: {n0 - len(pob):,} (el parquet ya viene filtrado)')

# Duplicados: misma empresa con mismos pros y cons (p. ej. reseña presente en los dos datasets)
n1 = len(pob)
pob = pob.drop_duplicates(['firm', 'pros', 'cons'])
print(f'Duplicados eliminados: {n1 - len(pob):,}')

def limpiar(txt):
    """Normaliza el texto: los saltos de línea suelen separar viñetas -> '; '. Se mantiene
    mayúscula/minúscula y puntuación porque el modelo de sentimiento y los embeddings las aprovechan."""
    t = re.sub(r'[\r\n]+\s*[-•*·]*\s*', '; ', str(txt))
    t = re.sub(r'^\s*[-•*·]+\s*', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip(' ;')

for col in ['pros', 'cons']:
    pob[col] = pob[col].map(limpiar)

# Filtro de idioma ligero: los modelos (DistilBERT SST-2 y MiniLM) son en inglés. Se descartan solo
# las reseñas donde dominan palabras funcionales de otros idiomas (alemán, español, francés, portugués...).
EN = re.compile(r"\b(the|and|to|of|is|are|in|for|with|not|no|good|great|work|working|people|you|very|"
                r"lot|team|pay|hours|life|balance|management|none|nothing|can|be|it|there)\b", re.I)
OTROS = re.compile(r"\b(und|der|das|nicht|sehr|mit|ist|ein|eine|que|los|las|muy|una|el|del|não|muito|"
                   r"les|des|très|avec|pas|het|een|niet|della|molto|il|la)\b", re.I)
def es_ingles(t):
    n_en, n_ot = len(EN.findall(t)), len(OTROS.findall(t))
    return not (n_ot >= 2 and n_ot > n_en)

ingles = (pob['pros'] + ' ' + pob['cons']).map(es_ingles)
print(f'Reseñas descartadas por no estar en inglés: {(~ingles).sum():,} ({100 * (~ingles).mean():.1f}%)')
pob = pob[ingles & (pob.pros.str.len() > 1) & (pob.cons.str.len() > 1)].reset_index(drop=True)

conteo = pob['firm'].value_counts().reindex(FIRMS)
print('\nReseñas válidas en la ventana por empresa:')
print(conteo.to_string())
assert (conteo >= MIN_RESENAS).all(), 'Alguna empresa no llega a 300 reseñas'
print(f'\nTodas superan las {MIN_RESENAS} reseñas: OK')

### 0.3 Variables derivadas
`current` mezcla situación y antigüedad (`"Former Employee, more than 3 years"`): se separa en **situación** (Actual/Antiguo) y **antigüedad** (tramos). `recommend` se traduce (v = Sí, o = Neutral/sin opinión, x = No).

In [ ]:
def situacion(s):
    s = str(s).lower()
    return 'Actual' if s.startswith('current') else 'Antiguo' if s.startswith('former') else 'Desconocido'

def antiguedad(s):
    m = re.search(r'(less|more) than (\d+) year', str(s))
    if not m:
        return 'No indicada'
    if m.group(1) == 'less':
        return '< 1 año'
    n = int(m.group(2))
    return '1-3 años' if n == 1 else '3-5 años' if n == 3 else '> 5 años'

ORDEN_ANTIG = ['< 1 año', '1-3 años', '3-5 años', '> 5 años', 'No indicada']

def preparar(d):
    d = d.copy()
    d['situacion'] = d['current'].map(situacion)
    d['antiguedad'] = pd.Categorical(d['current'].map(antiguedad), ORDEN_ANTIG, ordered=True)
    d['recomienda'] = d['recommend'].map({'v': 'Sí', 'o': 'Neutral', 'x': 'No'})
    d['anio'] = d['date_review'].dt.year
    d['trimestre'] = d['date_review'].dt.to_period('Q')
    d['periodo'] = np.where(d['anio'] < 2020, 'Antes de 2020', '2020 en adelante')
    return d

pob = preparar(pob)
print(pob['situacion'].value_counts().to_string(), '\n')
print(pob['antiguedad'].value_counts().reindex(ORDEN_ANTIG).to_string(), '\n')
print(pob['recomienda'].value_counts().to_string())

### 0.4 Muestra de trabajo para los modelos
El EDA se hace sobre **todas** las reseñas válidas (es barato y más preciso); los modelos de NLP, sobre una muestra estratificada por empresa de hasta 2.000 reseñas con semilla fija.

In [ ]:
df = (pd.concat([g.sample(n=min(N_MUESTRA, len(g)), random_state=SEED) for _, g in pob.groupby('firm')])
        .reset_index(drop=True))

# Clave de caché: si cambia la muestra (tamaño, semilla, filtros), se recalcula todo lo pesado
CLAVE = hashlib.md5(pd.util.hash_pandas_object(df[['firm', 'pros', 'cons']], index=False).values.tobytes()
                    ).hexdigest()[:10]
print(f'Muestra: {len(df):,} reseñas | clave de caché: {CLAVE}')

# ¿La muestra representa a la población? Rating medio y % actuales, muestra vs. total
repr_ = pd.DataFrame({
    'reseñas_total': pob.groupby('firm').size(), 'reseñas_muestra': df.groupby('firm').size(),
    'rating_total': pob.groupby('firm').overall_rating.mean(), 'rating_muestra': df.groupby('firm').overall_rating.mean(),
    '%actuales_total': pob.groupby('firm').situacion.apply(lambda s: 100 * (s == 'Actual').mean()),
    '%actuales_muestra': df.groupby('firm').situacion.apply(lambda s: 100 * (s == 'Actual').mean()),
}).reindex(FIRMS).round(2)
RES['muestra'] = repr_['reseñas_muestra'].astype(int).to_dict()
RES['poblacion'] = repr_['reseñas_total'].astype(int).to_dict()
repr_

---
## 1. EDA
### 1.1 Visión general por empresa
`% recomienda` se calcula **sobre quienes opinan** (Sí + No). La opción "sin opinión" crece mucho con los años en todo el dataset, así que incluirla en el denominador mezclaría un cambio del formulario de Glassdoor con un cambio real de opinión.

In [ ]:
resumen = pob.groupby('firm').agg(
    n_resenas=('firm', 'size'),
    rating_medio=('overall_rating', 'mean'),
    pct_recomienda=('recomienda', pct_recomienda),
    pct_si_sobre_total=('recomienda', lambda s: 100 * (s == 'Sí').mean()),
    pct_sin_opinion=('recomienda', lambda s: 100 * (s == 'Neutral').mean()),
    pct_actuales=('situacion', lambda s: 100 * (s == 'Actual').mean()),
).reindex(FIRMS)
display(resumen.round(2))

sin_op_anual = pob.groupby('anio')['recomienda'].apply(lambda s: 100 * (s == 'Neutral').mean()).round(1)
print('% de reseñas "sin opinión" por año (todas las empresas):', sin_op_anual.to_dict())

RES['eda'] = resumen.round(3).to_dict(orient='index')
pos_rating, pos_rec = posicion(resumen.rating_medio), posicion(resumen.pct_recomienda)
peers_rating = resumen.loc[PEERS, 'rating_medio'].mean()
peers_rec = resumen.loc[PEERS, 'pct_recomienda'].mean()

fig, axes = figura(
    f"Deloitte es {pos_rating}ª de 4 en rating ({resumen.loc[TARGET, 'rating_medio']:.2f} vs {peers_rating:.2f} de los peers) "
    f"y {pos_rec}ª en recomendación ({resumen.loc[TARGET, 'pct_recomienda']:.0f}% vs {peers_rec:.0f}%)",
    'Todas las reseñas válidas jul-2018 – jul-2023. % recomienda calculado sobre quienes opinan (Sí + No).',
    ncols=3, figsize=(15, 4.8))
paneles = [('rating_medio', 'Rating medio (1-5)', '{:.2f}', (3, 4.3)),
           ('pct_recomienda', '% que recomienda', '{:.0f}%', (0, 100)),
           ('pct_actuales', '% reseñas de empleados actuales', '{:.0f}%', (0, 100))]
for ax, (col, tit, fmt, lim) in zip(axes, paneles):
    vals = resumen[col][::-1]
    ax.barh(vals.index, vals.values, color=[COLORES[f] if f == TARGET else GRIS_PEERS for f in vals.index], height=0.6)
    ax.set_xlim(*lim); ax.set_title(tit); ax.grid(axis='y', visible=False)
    for i, v in enumerate(vals.values):
        ax.text(v + (lim[1] - lim[0]) * 0.015, i, fmt.format(v), va='center', fontsize=12, color=TINTA)
guardar(fig, '01_vision_general')

**Conclusión accionable (1.1).** Deloitte es la Big 4 mejor valorada: **1ª en rating (3,90 vs 3,80 de media de los peers)** y **1ª en recomendación (78,6% vs 75,9%)**, y es la que más reseñas tiene (40.187). La ventaja es real pero **estrecha frente a PwC** (3,87 y 77,5%). **Acción:** RRHH puede usar "la Big 4 mejor valorada por sus empleados en Glassdoor" en *employer branding*, pero debe vigilarlo cada trimestre porque PwC está a pocas centésimas. Además, la opción "sin opinión" en `recommend` pasa del 35% (2018) al 53% (2023), así que el KPI de recomendación debe medirse siempre sobre quienes opinan.

### 1.2 Empleados actuales vs. antiguos

In [ ]:
sit = (pob[pob.situacion != 'Desconocido']
       .groupby(['firm', 'situacion'])
       .agg(n=('firm', 'size'), rating=('overall_rating', 'mean'), pct_recomienda=('recomienda', pct_recomienda))
       .unstack('situacion').reindex(FIRMS))
sit[('brecha_rating', '')] = sit[('rating', 'Actual')] - sit[('rating', 'Antiguo')]
display(sit.round(2))
RES['actual_vs_antiguo'] = {f: {'rating_actual': sit.loc[f, ('rating', 'Actual')], 'rating_antiguo': sit.loc[f, ('rating', 'Antiguo')],
                                'rec_actual': sit.loc[f, ('pct_recomienda', 'Actual')], 'rec_antiguo': sit.loc[f, ('pct_recomienda', 'Antiguo')]}
                            for f in FIRMS}

brecha = sit[('brecha_rating', '')]
fig, axes = figura(
    f"Los antiguos empleados puntúan peor en las cuatro firmas (entre {brecha.min():.2f} y {brecha.max():.2f} puntos menos); "
    f"en Deloitte la brecha es de {brecha[TARGET]:.2f}",
    'Rating medio y % que recomienda según situación del empleado. Sesgo de selección esperable: quien se va suele estar menos satisfecho.',
    ncols=2, figsize=(14, 5))
x = np.arange(len(FIRMS)); w = 0.38
for ax, met, fmt, lim in [(axes[0], 'rating', '{:.2f}', (1, 5)), (axes[1], 'pct_recomienda', '{:.0f}%', (0, 100))]:
    for k, (s, alpha) in enumerate([('Actual', 1.0), ('Antiguo', 0.45)]):
        vals = sit[(met, s)].values
        cols = [COLORES[f] if f == TARGET else '#5F6B7A' for f in FIRMS]
        bars = ax.bar(x + (k - 0.5) * w, vals, w * 0.95, color=cols, alpha=alpha, label=s)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + (lim[1] - lim[0]) * 0.01, fmt.format(v), ha='center', va='bottom', fontsize=10.5)
    ax.set_xticks(x, FIRMS); ax.set_ylim(*lim); ax.grid(axis='x', visible=False)
    ax.set_title('Rating medio' if met == 'rating' else '% que recomienda')
axes[0].legend(['Actuales', 'Antiguos'], loc='upper right', ncols=2)
guardar(fig, '02_actuales_vs_antiguos')

**Conclusión accionable (1.2).** En las cuatro firmas los antiguos puntúan peor (sesgo de selección esperado). En Deloitte la brecha es la mayor (**3,99 actuales vs 3,69 antiguos, −0,31**; recomendación 82% vs 71%), pero no porque sus antiguos estén peor que los de la competencia (3,69, igual que PwC con 3,70), sino porque **sus empleados actuales están más contentos**. Los antiguos son el 30% de las reseñas de Deloitte y siguen opinando en público. **Acción:** cuidar la salida (entrevistas de salida estructuradas y red de *alumni*) es una palanca directa de reputación.

### 1.3 Evolución temporal del rating

In [ ]:
trim = pob.groupby(['trimestre', 'firm']).agg(rating=('overall_rating', 'mean'), n=('firm', 'size')).reset_index()
trim = trim[trim['n'] >= 50]                       # se descartan trimestres con muy pocas reseñas (bordes de la ventana)
anual = pob.groupby(['anio', 'firm']).overall_rating.mean().unstack().reindex(columns=FIRMS)
display(anual.round(2))
RES['rating_anual'] = anual.round(3).to_dict()

d19, d22 = anual.loc[2019, TARGET], anual.loc[2022, TARGET]
p19, p22 = anual.loc[2019, PEERS].mean(), anual.loc[2022, PEERS].mean()
verbo = lambda a, b: 'sube' if b > a + 0.02 else 'baja' if b < a - 0.02 else 'se mantiene'
fig, ax = figura(
    f"El rating de Deloitte {verbo(d19, d22)} de {d19:.2f} (2019) a {d22:.2f} (2022); el de los peers {verbo(p19, p22)} "
    f"de {p19:.2f} a {p22:.2f}",
    'Rating medio trimestral por empresa (trimestres con ≥ 50 reseñas). Línea discontinua: inicio de la pandemia (mar-2020).',
    figsize=(13, 5.5))
finales = {}
for f in FIRMS[::-1]:
    t = trim[trim.firm == f]
    xs = t['trimestre'].dt.to_timestamp()
    ax.plot(xs, t['rating'], color=COLORES[f], lw=3 if f == TARGET else 1.8, zorder=3 if f == TARGET else 2)
    finales[f] = (xs.iloc[-1], t['rating'].iloc[-1])
ys_etq = separar([finales[f][1] for f in FIRMS], 0.03)      # evita que las etiquetas se pisen
for f, ye in zip(FIRMS, ys_etq):
    ax.text(max(v[0] for v in finales.values()) + pd.Timedelta(days=30), ye, f, color=COLORES[f] if f == TARGET else TINTA2,
            va='center', fontsize=11.5, fontweight='bold' if f == TARGET else 'normal')
ax.axvline(pd.Timestamp('2020-03-01'), color=TINTA2, ls='--', lw=1)
ax.set_ylabel('Rating medio'); ax.grid(axis='x', visible=False)
guardar(fig, '03_evolucion_rating')

**Conclusión accionable (1.3).** El rating sube en todo el sector, pero Deloitte sube más: de **3,85 (2019) a 4,00 (2022)**, frente a 3,73 → 3,85 de los peers. Su liderazgo solo se interrumpió durante 2020-2021, cuando PwC la igualó o superó en algunos trimestres (la pandemia golpeó más a Deloitte en el 3T de 2020: 3,81). **Acción:** la ventaja se ha ampliado desde 2022 y es un buen momento para comunicarla; conviene montar un seguimiento trimestral de este indicador para detectar a tiempo si vuelve a cerrarse.

---
## 2. Sentimiento en pros y cons (control de calidad)

**Por qué este paso.** Los pros deberían ser positivos y los cons negativos. Si el modelo no lo confirma en la gran mayoría de casos, algo falla (modelo, truncado o limpieza) y no podríamos fiarnos de los textos para el resto del diagnóstico.

**Elección del modelo: `distilbert-base-uncased-finetuned-sst-2-english`.**
- *Negocio*: es un control de calidad, no el resultado final, así que prima un modelo estándar, auditable y rápido (se puede repetir cada trimestre en minutos). Es el modelo de sentimiento en inglés más usado y citado (Sanh et al., 2019), por lo que sus resultados son defendibles ante RRHH.
- *Técnica*: transformer (entiende negaciones y contexto: "no work-life balance" es negativo aunque contenga "balance"), a diferencia de un léxico tipo VADER. Es binario (POSITIVE/NEGATIVE, sin neutral): justo lo que queremos verificar. Las reseñas están en inglés (filtradas en 0.2).
- Máximo 512 tokens → `truncation=True`. Pros y cons se analizan **por separado**.

In [ ]:
from transformers import pipeline

MODELO_SENT = 'distilbert-base-uncased-finetuned-sst-2-english'
ruta_sent = CACHE_DIR / f'sentimiento_{CLAVE}.parquet'

if ruta_sent.exists():
    sent = pd.read_parquet(ruta_sent)
    print('Sentimiento cargado de caché:', ruta_sent)
else:
    clf = pipeline('sentiment-analysis', model=MODELO_SENT, device=DEVICE)
    sent = pd.DataFrame(index=df.index)
    for col in ['pros', 'cons']:
        t0 = time.time()
        textos = df[col].tolist()
        orden = np.argsort([len(t) for t in textos])          # agrupar por longitud acelera mucho en CPU
        salida = clf([textos[i] for i in orden], batch_size=64 if GPU else 32, truncation=True, max_length=512)
        etiquetas, conf = [None] * len(textos), [None] * len(textos)
        for i, o in zip(orden, salida):
            etiquetas[i], conf[i] = o['label'], o['score']
        sent[f'sent_{col}'], sent[f'conf_{col}'] = etiquetas, conf
        print(f'{col}: {len(textos):,} textos en {time.time() - t0:.0f} s')
    sent.to_parquet(ruta_sent)
df = df.join(sent)

calidad = df.groupby('firm').agg(
    pct_pros_positivos=('sent_pros', lambda s: 100 * (s == 'POSITIVE').mean()),
    pct_cons_negativos=('sent_cons', lambda s: 100 * (s == 'NEGATIVE').mean())).reindex(FIRMS)
calidad.loc['Total'] = [100 * (df.sent_pros == 'POSITIVE').mean(), 100 * (df.sent_cons == 'NEGATIVE').mean()]
RES['sentimiento'] = calidad.round(2).to_dict(orient='index')
calidad.round(1)

In [ ]:
fig, axes = figura(
    f"El modelo confirma la coherencia de los textos: {calidad.loc['Total', 'pct_pros_positivos']:.0f}% de los pros son positivos "
    f"y {calidad.loc['Total', 'pct_cons_negativos']:.0f}% de los cons negativos",
    'distilbert-base-uncased-finetuned-sst-2-english sobre pros y cons por separado (muestra de trabajo).',
    ncols=2, figsize=(13, 4.5))
for ax, col, tit in [(axes[0], 'pct_pros_positivos', '% de pros clasificados como POSITIVOS'),
                     (axes[1], 'pct_cons_negativos', '% de cons clasificados como NEGATIVOS')]:
    vals = calidad.loc[FIRMS, col][::-1]
    ax.barh(vals.index, vals.values, color=[VERDE if f == TARGET else GRIS_PEERS for f in vals.index], height=0.6)
    for i, v in enumerate(vals.values):
        ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=12)
    ax.set_xlim(0, 108); ax.set_title(tit); ax.grid(axis='y', visible=False)
guardar(fig, '04_control_sentimiento')

### 2.1 Casos discordantes: ¿qué son?
Se clasifican con reglas simples y transparentes para cuantificar de qué tipo son, y se leen ejemplos.

In [ ]:
SIN_QUEJAS = re.compile(r"^\W*(none|nothing|nil\b|n/?a\b|na\b|no\b|not (really|much|many|any|that i)|can'?t think|cannot think|"
                        r"i (can'?t|cannot|don'?t|do not) (think|see|find|have)|all good|so far so good|nope|zero|"
                        r"there (are|is) (no|none|nothing))", re.I)
MATIZ = re.compile(r"\b(but|however|although|though|sometimes|can be|could be|depends|may|might|at times|if)\b", re.I)
NEGACION = re.compile(r"\b(no|not|never|less|without|don'?t|isn'?t|aren'?t|lack|zero)\b", re.I)

def tipo_cons_positivo(t):
    if SIN_QUEJAS.search(t): return 'Sin quejas reales ("nothing", "none")'
    if MATIZ.search(t): return 'Queja suavizada o condicional ("can be", "but")'
    if len(t.split()) <= 6: return 'Frase corta o descriptiva sin carga negativa'
    return 'Queja redactada en tono positivo / sugerencia'

def tipo_pros_negativo(t):
    if SIN_QUEJAS.search(t): return 'Pro vacío ("nothing", "none")'
    if NEGACION.search(t): return 'Pro expresado con negación ("no micromanagement")'
    if MATIZ.search(t): return 'Elogio con matiz o contraste ("but")'
    if len(t.split()) <= 6: return 'Frase corta o neutra ("big company", "salary")'
    return 'Otros (ironía, mezcla de pros y cons)'

cons_pos = df[df.sent_cons == 'POSITIVE'].copy(); cons_pos['tipo'] = cons_pos['cons'].map(tipo_cons_positivo)
pros_neg = df[df.sent_pros == 'NEGATIVE'].copy(); pros_neg['tipo'] = pros_neg['pros'].map(tipo_pros_negativo)

for nombre, d, col in [('CONS clasificados como POSITIVOS', cons_pos, 'cons'), ('PROS clasificados como NEGATIVOS', pros_neg, 'pros')]:
    print(f'\n=== {nombre}: {len(d):,} ({100 * len(d) / len(df):.1f}% de la muestra) ===')
    tabla = d.groupby('tipo').agg(n=('tipo', 'size'), rating_medio=('overall_rating', 'mean'))
    tabla['%'] = 100 * tabla['n'] / tabla['n'].sum()
    print(tabla.sort_values('n', ascending=False).round(2).to_string())
    for tipo, g in d.groupby('tipo'):
        print(f'\n  -> {tipo}:')
        for _, r in g.sample(min(4, len(g)), random_state=SEED).iterrows():
            print(f"     [{r['firm']}, {r['overall_rating']:.0f}★] {r[col][:150]}")

RES['discordantes'] = {
    'cons_positivos_pct': 100 * len(cons_pos) / len(df), 'pros_negativos_pct': 100 * len(pros_neg) / len(df),
    'tipos_cons_pos': (100 * cons_pos['tipo'].value_counts(normalize=True)).round(1).to_dict(),
    'tipos_pros_neg': (100 * pros_neg['tipo'].value_counts(normalize=True)).round(1).to_dict()}
rating_sin_quejas = cons_pos.loc[cons_pos.tipo.str.startswith('Sin quejas'), 'overall_rating'].mean()
print(f"\nRating medio de reseñas 'sin quejas reales': {rating_sin_quejas:.2f} vs {df.overall_rating.mean():.2f} de media")
RES['discordantes']['rating_sin_quejas'] = rating_sin_quejas

**Lectura del control de calidad.** El **94,7% de los pros sale positivo** (93-95% por empresa) y el **81,3% de los cons sale negativo** (80-82%). Es coherente y homogéneo entre empresas: no hay un problema de datos que afecte a una firma más que a otra. Que los cons queden por debajo tiene explicación al leer los casos discordantes:

- **Cons "positivos" (18,7%)**:
  - 35%: frases telegráficas sin palabras negativas explícitas ("Challenging and long work hours", "Tough work life balance"). El modelo, entrenado con frases completas de críticas de cine (SST-2), falla con listas de sustantivos. Es una **limitación del modelo, no de los datos**.
  - 32%: quejas en tono constructivo o sugerencia.
  - 18%: quejas suavizadas ("can be", "sometimes").
  - 15%: **reseñas sin quejas reales** ("No cons, great place to work"). Aquí el modelo acierta: el texto es positivo, y estas reseñas tienen un rating de **4,61** frente a 3,82 de media. Se convierten en señal y aparecen como categoría propia ("Genérico / sin quejas").
- **Pros "negativos" (5,3%)**:
  - 23%: pros vacíos ("nothing", "no pros"), con un rating de **2,01**: son empleados realmente descontentos.
  - 18%: pros expresados con negación ("not that much workload").
  - El resto: elogios con contraste o ironía.

**Conclusión:** los textos son fiables para el análisis de temas. El modelo es suficiente como control de calidad. Un modelo con clase neutral o entrenado con reseñas cortas es una mejora para el futuro, no un bloqueo.

---
## 3. Topic modeling con BERTopic

**Por qué BERTopic** (y no LDA): los pros y cons son textos muy cortos (mediana ≈ 7 palabras). LDA cuenta palabras y funciona mal con textos cortos; BERTopic agrupa por **significado** (embeddings), así "long hours" y "overtime" acaban juntos. Para RRHH esto significa temas más limpios y menos revisión manual.

**Regla de comparabilidad:** un **único modelo** para los pros de las cuatro empresas y **otro** para los cons. Si se entrenara uno por empresa, los temas no coincidirían y no se podrían comparar.

**Parámetros y su porqué**

| Parámetro | Valor | Justificación de negocio / técnica |
|---|---|---|
| Embeddings | `all-MiniLM-L6-v2` | Modelo de frases en inglés rápido y probado; con textos cortos rinde casi como modelos 5× más pesados, y permite repetir el análisis cada trimestre en un portátil. |
| `min_topic_size` (HDBSCAN `min_cluster_size`) | 40 | Un tema con < 40 reseñas (0,5% de la muestra) no justifica una acción de RRHH; evita cientos de micro-temas. |
| `min_samples` | 10 | Más bajo que el tamaño mínimo para que menos reseñas queden como outlier. |
| UMAP | 15 vecinos, 5 dimensiones, `random_state=42` | Valores estándar de BERTopic; la semilla hace el resultado reproducible. |
| Vectorizador | Stopwords en inglés + nombres de las firmas, bigramas, `min_df=5` | Se quitan "deloitte", "pwc"... para que ningún tema sea "de una empresa" (rompería la comparación). Los bigramas capturan "work life", "long hours". Se conservan "no", "not", "none", "nothing" porque distinguen las reseñas "sin quejas". |
| `nr_topics` | `None` (sin fusión automática) | La agrupación en categorías se hace **a mano con criterio de RRHH** (abajo); la fusión automática puede mezclar temas que son palancas distintas (p. ej. salario y formación). |

In [ ]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

MODELO_EMB = 'all-MiniLM-L6-v2'
MIN_TOPIC_SIZE, MIN_SAMPLES = 40, 10

emb_model = SentenceTransformer(MODELO_EMB, device='cuda' if GPU else 'cpu')
EMB = {}
for col in ['pros', 'cons']:
    ruta = CACHE_DIR / f'emb_{col}_{CLAVE}.npy'
    if ruta.exists():
        EMB[col] = np.load(ruta)
    else:
        t0 = time.time()
        EMB[col] = emb_model.encode(df[col].tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
        np.save(ruta, EMB[col])
        print(f'Embeddings {col}: {time.time() - t0:.0f} s')
print({k: v.shape for k, v in EMB.items()})

STOPWORDS = sorted((set(ENGLISH_STOP_WORDS) - {'no', 'not', 'none', 'nothing', 'never', 'less', 'more', 'few', 'much', 'too'})
                   | {'deloitte', 'pwc', 'ey', 'kpmg', 'ernst', 'young', 'pricewaterhousecoopers', 'firm', 'company',
                      'good', 'great', 'nice', 'really', 'lot', 'lots', 'pros', 'cons', 'll', 've', 'don', 'isn', 't', 's'})

def entrenar_bertopic(col):
    """Entrena (o carga de caché) un BERTopic sobre `col` para las 4 empresas juntas."""
    ruta = CACHE_DIR / f'bertopic_{col}_{CLAVE}_{MIN_TOPIC_SIZE}_{MIN_SAMPLES}.pkl'
    if ruta.exists():
        with open(ruta, 'rb') as fh:
            return pickle.load(fh)
    t0 = time.time()
    modelo = BERTopic(
        embedding_model=emb_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED),
        hdbscan_model=HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, min_samples=MIN_SAMPLES, metric='euclidean',
                              cluster_selection_method='eom', prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words=STOPWORDS, ngram_range=(1, 2), min_df=5),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        top_n_words=10, calculate_probabilities=False, verbose=False)
    topics, _ = modelo.fit_transform(df[col].tolist(), EMB[col])
    res = {'topics': np.array(topics), 'modelo': modelo}
    with open(ruta, 'wb') as fh:
        pickle.dump(res, fh)
    print(f'BERTopic {col}: {time.time() - t0:.0f} s')
    return res

BT = {col: entrenar_bertopic(col) for col in ['pros', 'cons']}
for col in ['pros', 'cons']:
    t = BT[col]['topics']
    print(f'{col}: {len(set(t)) - (1 if -1 in t else 0)} topics | outliers (-1): {(t == -1).sum():,} ({100 * (t == -1).mean():.1f}%)')

### 3.1 Tratamiento de los outliers (topic −1)

HDBSCAN deja sin tema las reseñas que no encajan con claridad en ningún grupo. **Descartarlas no es neutro para el negocio:** suelen ser las reseñas más largas y con varios temas a la vez, es decir, las de los empleados que más se han explicado. Si las quitáramos, el diagnóstico se basaría solo en las reseñas telegráficas.

**Decisión:** reasignar cada outlier al tema cuyo centro (media de embeddings) sea **más parecido**, pero **solo si la similitud coseno supera un umbral**. Por debajo, la reseña queda como *"Sin clasificar"*: es preferible no etiquetar a etiquetar mal, porque una etiqueta forzada inflaría artificialmente alguna categoría y podría llevar a RRHH a invertir en la palanca equivocada. El umbral se fija mirando la similitud que tienen las reseñas que **sí** asignó HDBSCAN.

In [ ]:
def centroides(emb, topics):
    ids = sorted(t for t in set(topics) if t != -1)
    C = np.vstack([emb[topics == t].mean(axis=0) for t in ids])
    return ids, C / np.linalg.norm(C, axis=1, keepdims=True)

SIM_MAX = {}
for col in ['pros', 'cons']:
    t, E = BT[col]['topics'], EMB[col]
    ids, C = centroides(E, t)
    sims = E @ C.T                                   # embeddings normalizados -> producto = coseno
    SIM_MAX[col] = pd.DataFrame({'sim': sims.max(1), 'mejor': np.array(ids)[sims.argmax(1)], 'outlier': t == -1})
    q = SIM_MAX[col].groupby('outlier')['sim'].describe(percentiles=[.1, .25, .5])
    print(f'\n{col}: similitud con el tema más cercano (asignados vs outliers)')
    print(q.round(3).to_string())

# Umbral = percentil 10 de la similitud de las reseñas que HDBSCAN sí asignó: un outlier se reasigna
# solo si se parece a su tema al menos tanto como el 90% de los miembros "legítimos" de los temas.
UMBRAL = {col: float(SIM_MAX[col].loc[~SIM_MAX[col].outlier, 'sim'].quantile(0.10)) for col in ['pros', 'cons']}
print('\nUmbral de reasignación:', {k: round(v, 3) for k, v in UMBRAL.items()})

for col in ['pros', 'cons']:
    s = SIM_MAX[col]
    final = BT[col]['topics'].copy()
    reasignar = s.outlier & (s.sim >= UMBRAL[col])
    final[reasignar.values] = s.loc[reasignar, 'mejor'].values
    BT[col]['topics_final'] = final
    # Se recalculan las palabras clave con las asignaciones nuevas
    BT[col]['modelo'].update_topics(df[col].tolist(), topics=final.tolist(),
                                    vectorizer_model=CountVectorizer(stop_words=STOPWORDS, ngram_range=(1, 2), min_df=5),
                                    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True))
    print(f'{col}: outliers {(s.outlier).mean() * 100:.1f}% -> reasignados {reasignar.mean() * 100:.1f}% -> '
          f'sin clasificar {(final == -1).mean() * 100:.1f}%')
    df[f'topic_{col}'] = final

RES['outliers'] = {col: {'inicial_pct': 100 * (BT[col]['topics'] == -1).mean(),
                         'final_pct': 100 * (BT[col]['topics_final'] == -1).mean(), 'umbral': UMBRAL[col],
                         'n_topics': int(len(set(BT[col]['topics_final'])) - 1)} for col in ['pros', 'cons']}

### 3.2 De topics a categorías de negocio

Los topics son demasiados y demasiado finos para RRHH. Se agrupan en **10 categorías de negocio** (más *Genérico / sin quejas*, *Otros* y *Sin clasificar*), cada una asociada a una **palanca de gestión distinta** (quién en RRHH actuaría sobre ella). Por eso, por ejemplo, *conciliación* (política de flexibilidad) se separa de *carga de trabajo* (dimensionamiento de equipos), aunque los empleados las mencionen juntas.

**Método en dos pasos:**
1. **Propuesta automática:** cada topic se asigna a la categoría cuyas palabras clave coinciden más con sus 10 términos principales (ponderando más los primeros).
2. **Revisión manual:** se leen las palabras clave y las reseñas más representativas de **cada** topic y se corrigen las asignaciones erróneas en `REVISION_MANUAL`. Cada corrección incluye una palabra de control: si al re-ejecutar (p. ej. en otra máquina) el topic con ese número ya no contiene esa palabra, la corrección no se aplica y se avisa.

In [ ]:
CATEGORIAS = {
    'Compensación y beneficios': ['salary', 'salaries', 'pay', 'paid', 'compensation', 'bonus', 'bonuses', 'benefits', 'benefit',
                                  'raise', 'raises', 'hike', 'increment', 'money', 'perks', 'insurance', 'pension', 'wage',
                                  'wages', 'remuneration', 'package', 'underpaid', 'compensated', 'allowance', 'pto'],
    'Conciliación y flexibilidad': ['work life', 'life balance', 'balance', 'wlb', 'flexible', 'flexibility', 'hybrid', 'remote',
                                    'home', 'wfh', 'leave', 'holidays', 'holiday', 'vacation', 'personal life', 'family'],
    'Carga de trabajo y horarios': ['hours', 'long hours', 'workload', 'overtime', 'busy season', 'busy', 'pressure', 'stress',
                                    'stressful', 'deadlines', 'deadline', 'burnout', 'weekends', 'weekend', 'late', 'understaffed',
                                    'overworked', 'utilization', 'utilisation', 'chargeable', 'targets', 'pace', 'demanding'],
    'Dirección y management': ['management', 'manager', 'managers', 'partners', 'partner', 'leadership', 'leaders', 'senior',
                               'seniors', 'boss', 'micromanagement', 'micro', 'supervisor', 'directors', 'support management'],
    'Carrera y promoción': ['career', 'growth', 'promotion', 'promotions', 'progression', 'promoted', 'advancement',
                            'opportunities', 'opportunity', 'mobility', 'path', 'grow', 'recognition', 'appraisal', 'rating', 'performance'],
    'Aprendizaje y proyectos': ['learning', 'learn', 'training', 'trainings', 'exposure', 'experience', 'knowledge', 'skills',
                                 'clients', 'client', 'projects', 'project', 'variety', 'diverse projects', 'industries',
                                 'boring', 'repetitive', 'monotonous', 'mundane', 'interesting',
                                 'development', 'certification', 'qualification', 'cpa', 'acca', 'aca', 'study', 'technical'],
    'Cultura y compañeros': ['culture', 'people', 'colleagues', 'team', 'teams', 'environment', 'friendly', 'supportive',
                             'inclusive', 'inclusion', 'diversity', 'politics', 'political', 'atmosphere', 'coworkers',
                             'smart', 'collaborative', 'toxic', 'values', 'social', 'events'],
    'Marca y prestigio': ['brand', 'name', 'big', 'reputation', 'prestige', 'prestigious', 'cv', 'resume', 'network',
                          'networking', 'global', 'recognized', 'recognised', 'international', 'big4'],
    'Procesos y organización': ['process', 'processes', 'bureaucracy', 'bureaucratic', 'admin', 'administrative', 'systems',
                                'tools', 'technology', 'slow', 'structure', 'hierarchy', 'hierarchical', 'red tape',
                                'size', 'large', 'organization', 'organisation', 'communication', 'silos', 'internal', 'change'],
    'Rotación y estabilidad': ['job security', 'security', 'stable', 'stability', 'layoffs', 'layoff', 'redundancy',
                            'redundancies', 'secure', 'cuts', 'fired', 'turnover', 'attrition'],
    'Genérico / sin quejas': ['none', 'nothing', 'think', 'mention', 'complaints', 'far', 'specific', 'say', 'everything', 'cons'],
}
CATS_SUST = [c for c in CATEGORIAS if c != 'Genérico / sin quejas']     # categorías "de negocio"
SIN_CLASIFICAR = 'Sin clasificar'

def proponer_categoria(palabras):
    """Puntuación por categoría: coincidencia de cada término clave (ponderado por su posición)."""
    puntos = {c: 0.0 for c in CATEGORIAS}
    for i, w in enumerate(palabras[:10]):
        for c, kws in CATEGORIAS.items():
            if any(re.search(rf'\b{re.escape(k)}\b', w) for k in kws):
                puntos[c] += 1 / (1 + i)
    mejor = max(puntos, key=puntos.get)
    return mejor if puntos[mejor] > 0 else 'Otros'

# Correcciones tras la revisión manual: {col: {topic_id: (categoría, palabra de control)}}
# Motivo de cada corrección en el comentario; la palabra de control protege frente a cambios de numeración.
REVISION_MANUAL = {
    'pros': {
        3:  ('Genérico / sin quejas', 'place'),              # "great place to work": elogio genérico, no habla de carrera
        6:  ('Genérico / sin quejas', 'work work'),          # "good company to work for"
        12: ('Marca y prestigio', 'respected'),              # "large, respected, prestigious": prestigio, no procesos
        15: ('Aprendizaje y proyectos', 'audit'),            # formación y experiencia en auditoría/contabilidad
        17: ('Conciliación y flexibilidad', 'flexible'),     # "flexible working hours": flexibilidad, no carga de trabajo
        24: ('Compensación y beneficios', 'office'),         # oficinas, café, instalaciones: beneficios tangibles
        25: ('Genérico / sin quejas', 'wonderful'),          # "great to work at ...", sin tema concreto
        26: ('Carrera y promoción', 'college'),              # "great place to start/launch a career" para recién graduados
        31: ('Aprendizaje y proyectos', 'fast paced'),       # trabajo retador y variado
        32: ('Compensación y beneficios', 'pto'),            # vacaciones pagadas = política de beneficios
        41: ('Genérico / sin quejas', 'social events'),      # elogios genéricos a la firma
        44: ('Compensación y beneficios', 'vacation'),       # vacaciones pagadas = política de beneficios
    },
    'cons': {
        1:  ('Compensación y beneficios', 'low pay'),        # "long hours and low pay": lo distintivo es que el sueldo no compensa
        12: ('Carga de trabajo y horarios', 'client'),       # "difficult/demanding clients": presión de cliente, no aprendizaje
        17: ('Compensación y beneficios', 'big4'),           # "less pay compared to other Big 4": queja salarial
        19: ('Carga de trabajo y horarios', 'audit'),        # horas largas en temporada de auditoría/impuestos
        28: ('Rotación y estabilidad', 'turnover'),          # rotación de plantilla
        29: ('Aprendizaje y proyectos', 'boring'),           # trabajo repetitivo/aburrido: calidad de los proyectos
        43: ('Genérico / sin quejas', 'professionals'),      # "nothing to share as cons", "couldn't find any cons"
    },
}

def tabla_topics(col):
    modelo, final, E = BT[col]['modelo'], BT[col]['topics_final'], EMB[col]
    filas = []
    for t in sorted(set(final)):
        n = int((final == t).sum())
        if t == -1:
            filas.append({'topic': -1, 'palabras_clave': '(outliers no reasignados)', 'categoria_auto': SIN_CLASIFICAR,
                          'categoria': SIN_CLASIFICAR, 'n': n, 'ejemplos': ''})
            continue
        palabras = [w for w, _ in modelo.get_topic(t)][:10]
        auto = proponer_categoria(palabras)
        cat = auto
        if t in REVISION_MANUAL[col]:
            cat_man, control = REVISION_MANUAL[col][t]
            if any(control in w for w in palabras):
                cat = cat_man
            else:
                print(f'AVISO {col} topic {t}: palabra de control "{control}" no encontrada; se mantiene "{auto}"')
        # Reseñas más representativas = más cercanas al centro del topic (cortas, para leerlas rápido)
        idx = np.where(final == t)[0]
        c = E[idx].mean(0); c /= np.linalg.norm(c)
        cerca = idx[np.argsort(-(E[idx] @ c))][:3]
        filas.append({'topic': t, 'palabras_clave': ', '.join(palabras[:7]), 'categoria_auto': auto, 'categoria': cat,
                      'n': n, 'ejemplos': ' | '.join(df.loc[i, col][:70] for i in cerca)})
    tab = pd.DataFrame(filas)
    tab['%'] = 100 * tab['n'] / len(df)
    return tab

TOPICS = {col: tabla_topics(col) for col in ['pros', 'cons']}
for col in ['pros', 'cons']:
    mapa = dict(zip(TOPICS[col]['topic'], TOPICS[col]['categoria']))
    df[f'cat_{col}'] = df[f'topic_{col}'].map(mapa)

**Tabla de topics de los PROS** (topic → palabras clave → categoría → nº y % de reseñas). La columna `ejemplos` muestra las tres reseñas más representativas, base de la revisión manual.

In [ ]:
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 230):
    display(TOPICS['pros'].round(1))

**Tabla de topics de los CONS**

In [ ]:
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 230):
    display(TOPICS['cons'].round(1))

**Revisión manual.** El modelo final tiene **47 topics en pros y 50 en cons**. Los outliers bajan del 28,8% al **8,8% (pros) y 9,7% (cons)**, que quedan como "Sin clasificar".

**Correcciones:** la propuesta automática acertó en la mayoría de topics. Se corrigieron **12 en pros y 7 en cons**; las más relevantes:
- *"less pay compared to other Big 4"*: la palabra "Big 4" lo llevaba a Marca, pero es una **queja salarial**.
- *"flexible working hours"*: lo llevaba a Carga de trabajo, pero es **flexibilidad**.
- *"long hours and low pay"*: lo distintivo es que el sueldo no compensa, así que va a **Compensación**.
- *"difficult/demanding clients"*: es **presión de cliente**, no aprendizaje.
- *"nothing to share as cons"*: va a **sin quejas**.
- *"great place to work"* y *"good company to work for"*: son **elogio genérico**, no carrera.

**Observación honesta:** aunque se quitaron los nombres de las firmas del vocabulario, los embeddings crean algún topic "de empresa":
- *"EY is a great place to start your career"* (pros, Carrera): por eso EY tiene un 7,2% de pros de carrera frente al ~3% del resto.
- *"PwC is a great company"* (pros, Genérico).
- *"no cons at Deloitte"* (cons, Genérico).

Se han asignado a su categoría de negocio, pero inflan algo esas categorías en esa empresa y se tienen en cuenta al interpretar.

*Carga de trabajo* solo aparece en cons y *Marca y prestigio* solo en pros, así que en ellas el balance neto refleja un único lado.

In [ ]:
ORDEN_CATS = CATS_SUST + ['Genérico / sin quejas', 'Otros', SIN_CLASIFICAR]
resumen_cat = pd.DataFrame({
    'topics_pros': TOPICS['pros'].groupby('categoria').size(), 'topics_cons': TOPICS['cons'].groupby('categoria').size(),
    '%_pros': 100 * df['cat_pros'].value_counts(normalize=True), '%_cons': 100 * df['cat_cons'].value_counts(normalize=True),
}).reindex([c for c in ORDEN_CATS if c in set(df.cat_pros) | set(df.cat_cons)]).fillna(0)
RES['categorias'] = resumen_cat.round(2).to_dict(orient='index')
RES['n_categorias_negocio'] = int(sum(c in CATS_SUST for c in resumen_cat.index))
display(resumen_cat.round(1))

top_pro = resumen_cat.loc[CATS_SUST, '%_pros'].idxmax(); top_con = resumen_cat.loc[CATS_SUST, '%_cons'].idxmax()
fig, ax = figura(
    f"En las Big 4 lo que más se elogia es «{top_pro}» y de lo que más se quejan, «{top_con}»",
    '% de reseñas (4 empresas, muestra de trabajo) cuyo texto principal cae en cada categoría.', figsize=(12, 6.5))
orden = resumen_cat.loc[[c for c in resumen_cat.index if c != 'Otros']].sort_values('%_cons').index
y = np.arange(len(orden)); h = 0.38
ax.barh(y + h / 2, resumen_cat.loc[orden, '%_pros'], h * 0.95, color=VERDE_SUAVE, label='Pros')
ax.barh(y - h / 2, resumen_cat.loc[orden, '%_cons'], h * 0.95, color=ROJO_SUAVE, label='Cons')
ax.set_yticks(y, orden); ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.legend(loc='lower right'); ax.grid(axis='y', visible=False)
etiquetar_barras(ax, '{:.1f}%', size=10)
guardar(fig, '05_categorias_global')

---
## 4. Comparación con peers

Para cada empresa se calcula el % de sus reseñas cuyos pros (o cons) caen en cada categoría, y se compara Deloitte con la **media simple de PwC, EY y KPMG** (cada competidor pesa lo mismo).

> **Cuidado con los porcentajes.** Los porcentajes de cada empresa suman 100%, así que son **relativos**: si una empresa tiene pocas quejas de salario, el peso de "salario" en sus cons baja, pero sube el del resto aunque nada haya empeorado. Un % alto en cons no significa por sí solo "estamos peor". Por eso la métrica principal es el **balance neto**:
>
> **balance neto = % de reseñas que la destacan en pros − % que la mencionan en cons**
>
> y su **gap** frente a la media de los peers (en puntos porcentuales, pp). Gap > 0 → fortaleza relativa; gap < 0 → debilidad relativa.

Para no confundir ruido con señal, cada gap lleva un **test de significación** (z, 95%): con ~2.000 reseñas por empresa, diferencias de 2-3 pp pueden ser azar.

In [ ]:
def tabla_pct(d, col):
    return (pd.crosstab(d['firm'], d[col], normalize='index') * 100).reindex(index=FIRMS, columns=ORDEN_CATS).fillna(0)

pct_pros, pct_cons = tabla_pct(df, 'cat_pros'), tabla_pct(df, 'cat_cons')
balance = pct_pros - pct_cons

def gap_con_z(d, cat, modo='balance'):
    """Gap Deloitte - media peers y su z. Por reseña: x = 1[pros en cat] - 1[cons en cat] (balance) o 1[cons en cat]."""
    x = (d['cat_pros'].eq(cat).astype(int) - d['cat_cons'].eq(cat).astype(int)) if modo == 'balance' else d['cat_cons'].eq(cat).astype(int)
    xd, xp = x[d.firm == TARGET], x[d.firm.isin(PEERS)]
    medias_peers = [x[d.firm == p].mean() for p in PEERS]
    gap = 100 * (xd.mean() - np.mean(medias_peers))
    se = 100 * math.sqrt(xd.var(ddof=1) / len(xd) + xp.var(ddof=1) / len(xp)) if len(xd) > 1 and len(xp) > 1 else np.nan
    return gap, (gap / se if se and se > 0 else 0.0)

comp = pd.DataFrame(index=CATS_SUST)
comp['pros_Deloitte'] = pct_pros.loc[TARGET, CATS_SUST]
comp['pros_peers'] = pct_pros.loc[PEERS, CATS_SUST].mean()
comp['cons_Deloitte'] = pct_cons.loc[TARGET, CATS_SUST]
comp['cons_peers'] = pct_cons.loc[PEERS, CATS_SUST].mean()
comp['balance_Deloitte'] = comp['pros_Deloitte'] - comp['cons_Deloitte']
comp['balance_peers'] = comp['pros_peers'] - comp['cons_peers']
gz = {c: gap_con_z(df, c) for c in CATS_SUST}
comp['gap_balance'] = [gz[c][0] for c in CATS_SUST]
comp['z'] = [gz[c][1] for c in CATS_SUST]
comp['significativo'] = comp['z'].abs() >= 1.96
comp = comp.sort_values('gap_balance', ascending=False)
RES['comparacion'] = comp.round(3).to_dict(orient='index')

print('Peso de cada categoría por empresa (% de reseñas) — PROS')
display(pct_pros.round(1))
print('CONS')
display(pct_cons.round(1))
comp.round(2)

In [ ]:
fig, axes = figura(
    'Dónde se diferencia Deloitte: peso de cada categoría en pros y en cons frente a la media de los peers',
    '% de reseñas de cada empresa. Punto verde = Deloitte; punto gris = media PwC/EY/KPMG.',
    ncols=2, figsize=(15, 6.5), sharey=True)
orden = comp.sort_values('cons_Deloitte').index
y = np.arange(len(orden))
for ax, tipo, tit in [(axes[0], 'pros', 'En PROS (lo que se elogia)'), (axes[1], 'cons', 'En CONS (de lo que se quejan)')]:
    d, p = comp.loc[orden, f'{tipo}_Deloitte'], comp.loc[orden, f'{tipo}_peers']
    ax.hlines(y, np.minimum(d, p), np.maximum(d, p), color='#D5D9DE', lw=4, zorder=1)
    ax.scatter(p, y, s=90, color=GRIS_PEERS, zorder=2, label='Media peers')
    ax.scatter(d, y, s=110, color=VERDE, zorder=3, label='Deloitte')
    for yi, (dv, pv) in enumerate(zip(d, p)):
        ax.text(max(dv, pv) + 0.6, yi, f'{dv - pv:+.1f} pp', va='center', fontsize=10, color=TINTA2)
    ax.set_title(tit); ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0)); ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, orden); axes[0].legend(loc='lower right')
guardar(fig, '06_peso_pros_cons_vs_peers')

In [ ]:
mejor, peor = comp['gap_balance'].idxmax(), comp['gap_balance'].idxmin()
fig, ax = figura(
    f"Deloitte saca más ventaja en «{mejor}» ({comp.loc[mejor, 'gap_balance']:+.1f} pp) y queda más por detrás en «{peor}» "
    f"({comp.loc[peor, 'gap_balance']:+.1f} pp)",
    'Gap del balance neto (% en pros − % en cons) de Deloitte frente a la media de los peers, en puntos porcentuales. '
    'Barras claras: diferencia no significativa al 95%.', figsize=(12.5, 6.2))
orden = comp.sort_values('gap_balance').index
vals = comp.loc[orden, 'gap_balance']
cols = [(VERDE if v > 0 else ROJO) if s else (VERDE_SUAVE if v > 0 else ROJO_SUAVE)
        for v, s in zip(vals, comp.loc[orden, 'significativo'])]
ax.barh(orden, vals, color=cols, height=0.62)
ax.axvline(0, color=TINTA2, lw=1)
lim = max(abs(vals).max() * 1.35, 3)
ax.set_xlim(-lim, lim)
for i, v in enumerate(vals):
    ax.text(v + (0.15 if v >= 0 else -0.15), i, f'{v:+.1f} pp', va='center', ha='left' if v >= 0 else 'right', fontsize=11)
ax.set_xlabel('← peor que los peers      |      mejor que los peers →'); ax.grid(axis='y', visible=False)
guardar(fig, '07_gap_balance_neto')

**Lectura del balance neto.**
- **Fortalezas relativas (significativas):**
  - **Compensación y beneficios, +6,5 pp:** Deloitte tiene más pros de salario y beneficios (11,8% vs 9,4%) y menos quejas (15,2% vs 19,3%).
  - **Carga de trabajo y horarios, +5,9 pp:** sigue siendo su primera queja (20,6%), pero menos que en los peers (26,5%).
- **En línea con el mercado:** cultura, dirección, conciliación, procesos y rotación (gaps < 1 pp, no significativos).
- **Debilidades relativas (significativas):**
  - **Aprendizaje y proyectos, −3,4 pp:** menos elogios (19,8% vs 21,2%) y más quejas (6,7% vs 4,8%), sobre todo de proyectos repetitivos, de "quedarse atascado" en un proyecto y de poca formación.
  - **Marca y prestigio, −2,4 pp:** la marca se elogia menos (6,8% vs 9,3%).
  - **Carrera y promoción, −2,4 pp.** Ojo: la parte de pros está inflada por el topic propio de EY. El dato robusto es el de cons: Deloitte tiene un 4,9% de quejas de promoción frente al 3,7% de los peers.

Recordatorio: el peso de una categoría en cons no basta para decir "estamos peor". Por ejemplo, Deloitte tiene un 12,0% de cons "sin quejas" frente al ~7,9% de los peers, y eso reduce el peso de todas las demás categorías.

### 4.1 Contraste con `overall_rating` y `recommend`
Si una categoría es de verdad un problema, las reseñas que se quejan de ella deberían tener **peor rating** y **menos recomendación** que la media.

In [ ]:
def contraste(d):
    t = d[d.cat_cons.isin(CATS_SUST + ['Genérico / sin quejas'])].groupby('cat_cons').agg(
        n=('firm', 'size'), rating=('overall_rating', 'mean'), pct_recomienda=('recomienda', pct_recomienda))
    return t

ctr_all = contraste(df).add_suffix('_todas')
ctr_d = contraste(df[df.firm == TARGET]).add_suffix('_Deloitte')
ctr = ctr_all.join(ctr_d).sort_values('rating_todas')
media_rating, media_rec = df.overall_rating.mean(), pct_recomienda(df.recomienda)
media_rating_d, media_rec_d = df[df.firm == TARGET].overall_rating.mean(), pct_recomienda(df[df.firm == TARGET].recomienda)
print(f'Media general: rating {media_rating:.2f} | recomienda {media_rec:.1f}%   (Deloitte: {media_rating_d:.2f} | {media_rec_d:.1f}%)')
RES['contraste_rating'] = ctr.round(3).to_dict(orient='index')
RES['media_rating_muestra'] = {'todas': media_rating, 'Deloitte': media_rating_d, 'rec_todas': media_rec, 'rec_Deloitte': media_rec_d}
display(ctr.round(2))

c_sust = ctr.drop(index='Genérico / sin quejas', errors='ignore')
p1, p2 = c_sust.index[0], c_sust.index[1]
fig, axes = figura(
    f"Quejarse de «{p1}» o «{p2}» va asociado al peor rating ({c_sust.loc[p1, 'rating_todas']:.2f} y "
    f"{c_sust.loc[p2, 'rating_todas']:.2f} vs {media_rating:.2f} de media): son problemas reales, no ruido",
    'Rating medio y % que recomienda de las reseñas según la categoría de su queja principal. Línea: media de la muestra.',
    ncols=2, figsize=(15, 6.3), sharey=True)
orden = ctr.sort_values('rating_todas', ascending=False).index
y = np.arange(len(orden))
for ax, met, media, fmt, tit in [(axes[0], 'rating', media_rating, '{:.2f}', 'Rating medio'),
                                 (axes[1], 'pct_recomienda', media_rec, '{:.0f}%', '% que recomienda')]:
    ax.scatter(ctr.loc[orden, f'{met}_todas'], y, s=90, color=GRIS_PEERS, label='Las 4 empresas', zorder=2)
    ax.scatter(ctr.loc[orden, f'{met}_Deloitte'], y, s=90, color=VERDE, marker='D', label='Solo Deloitte', zorder=3)
    ax.axvline(media, color=TINTA2, ls='--', lw=1)
    ax.set_title(tit); ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, orden); axes[0].legend(loc='lower left')
guardar(fig, '08_contraste_rating')

**Lectura del contraste.** Que una queja sea frecuente no significa que sea grave:
- **Dirección y management** (rating 3,29, 53% recomienda) y **Cultura y compañeros** (3,31, 59%) son las quejas que **más hunden la valoración**, muy por debajo de la media (3,82 y 77%). En Deloitte pasa lo mismo, aunque algo menos acusado (3,41 y 3,51).
- **Carga de trabajo** es la queja más frecuente, pero quienes la mencionan puntúan **por encima de la media (3,96, 84% recomienda)**. Es el "peaje aceptado" de trabajar en una Big 4: molesta, pero no rompe la relación.
- Las reseñas que se quejan de **Aprendizaje y proyectos** también puntúan por encima de la media (3,93; en Deloitte 4,04). Es una debilidad competitiva, pero de baja gravedad emocional: un riesgo de desgaste y de salida a medio plazo, no de reputación inmediata.

**Implicación:** management y cultura son "higiénicos". Deloitte está en línea con el mercado y debe mantenerlo, porque cuando fallan destruyen la recomendación (ver 7.3).

---
## 5. Empleados actuales vs. antiguos

Los cons de quienes ya se han ido son, en la práctica, **motivos de salida**: el input más directo para un plan de retención. Como los antiguos puntúan peor en todas las empresas (sesgo de selección, ver 1.2), aquí **no se compara sentimiento sino temas**: de qué se quejan unos y otros.

In [ ]:
dD = df[(df.firm == TARGET) & (df.situacion != 'Desconocido')]
act_ant = (pd.crosstab(dD['situacion'], dD['cat_cons'], normalize='index') * 100).reindex(columns=CATS_SUST).fillna(0).T
act_ant['dif_antiguo_menos_actual'] = act_ant['Antiguo'] - act_ant['Actual']
print('Deloitte — reseñas por situación:', dD.situacion.value_counts().to_dict())
display(act_ant.sort_values('Antiguo', ascending=False).round(1))

# Motivos de salida: antiguos de Deloitte vs antiguos de los peers
ant = df[df.situacion == 'Antiguo']
salida = pd.DataFrame(index=CATS_SUST)
salida['Deloitte'] = (pd.crosstab(ant.firm, ant.cat_cons, normalize='index') * 100).reindex(columns=CATS_SUST).fillna(0).loc[TARGET]
salida['media_peers'] = (pd.crosstab(ant.firm, ant.cat_cons, normalize='index') * 100).reindex(columns=CATS_SUST).fillna(0).loc[PEERS].mean()
gz_ant = {c: gap_con_z(ant, c, modo='cons') for c in CATS_SUST}
salida['gap_pp'] = [gz_ant[c][0] for c in CATS_SUST]
salida['z'] = [gz_ant[c][1] for c in CATS_SUST]
salida['significativo'] = salida['z'].abs() >= 1.96
salida = salida.sort_values('Deloitte', ascending=False)
print('Antiguos por empresa:', ant.firm.value_counts().reindex(FIRMS).to_dict())
RES['salida'] = salida.round(3).to_dict(orient='index')
RES['act_ant_deloitte'] = act_ant.round(3).to_dict(orient='index')
display(salida.round(2))

In [ ]:
top_salida = salida.index[0]
mas_que_peers = salida['gap_pp'].idxmax()
fig, axes = figura(
    f"El primer motivo de salida en Deloitte es «{top_salida}» ({salida.loc[top_salida, 'Deloitte']:.0f}% de los antiguos); "
    f"frente a los peers, destaca «{mas_que_peers}» ({salida.loc[mas_que_peers, 'gap_pp']:+.1f} pp)",
    'Izquierda: cons de empleados actuales vs antiguos de Deloitte. Derecha: cons de antiguos de Deloitte vs antiguos de los peers '
    '(pp; barras claras = no significativo).', ncols=2, figsize=(15.5, 6.5), sharey=True)
orden = salida.sort_values('Deloitte').index
y = np.arange(len(orden)); h = 0.38
axes[0].barh(y + h / 2, act_ant.loc[orden, 'Actual'], h * 0.95, color=VERDE, label='Actuales')
axes[0].barh(y - h / 2, act_ant.loc[orden, 'Antiguo'], h * 0.95, color=VERDE, alpha=0.45, label='Antiguos')
axes[0].set_yticks(y, orden); axes[0].legend(loc='lower right'); axes[0].grid(axis='y', visible=False)
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(5)); axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
axes[0].set_title('Deloitte: % de cons por categoría')
vals = salida.loc[orden, 'gap_pp']
cols = [(ROJO if v > 0 else VERDE) if s else (ROJO_SUAVE if v > 0 else VERDE_SUAVE) for v, s in zip(vals, salida.loc[orden, 'significativo'])]
axes[1].barh(y, vals, color=cols, height=0.6); axes[1].axvline(0, color=TINTA2, lw=1)
lim = max(abs(vals).max() * 1.4, 3); axes[1].set_xlim(-lim, lim)
for i, v in enumerate(vals):
    axes[1].text(v + (0.15 if v >= 0 else -0.15), i, f'{v:+.1f}', va='center', ha='left' if v >= 0 else 'right', fontsize=10.5)
axes[1].set_title('Antiguos: Deloitte − media peers (pp)'); axes[1].grid(axis='y', visible=False)
axes[1].set_xlabel('← se menciona menos que en peers  |  más que en peers →')
guardar(fig, '09_motivos_salida')

**Lectura actuales vs. antiguos.** Dentro de Deloitte, los antiguos se quejan más que los actuales de **Cultura (8,1% vs 4,7%)**, **Dirección (7,4% vs 4,4%)** y **Aprendizaje y proyectos (7,9% vs 6,2%)**. Son temas relacionales y de desarrollo.

Comparados con los antiguos de los peers (los motivos de salida):
- **Carga de trabajo pesa 6,4 pp menos** (significativo). En Deloitte la carga empuja menos a irse que en la competencia.
- **Aprendizaje y proyectos pesa 2,7 pp más** (significativo). Es el **motivo de salida diferencial** de Deloitte.
- **Carrera y promoción pesa 1,8 pp más.** Está al límite de la significación (z = 1,77), así que es un indicio que hay que vigilar.

Tamaños: 555 antiguos en Deloitte y 633-729 por peer. Se comparan temas, no tono, por el sesgo de selección ya visto en 1.2.

---
## 6. Áreas de mejora y recomendaciones
### 6.1 Matriz de priorización
- **Eje X — peso:** % de los cons de Deloitte que caen en la categoría (¿cuánto se habla de ello?).
- **Eje Y — desventaja frente a peers:** gap del balance neto con signo invertido (pp). Arriba = Deloitte está **peor** que el mercado.
- **Cuadrante prioritario** (arriba-derecha): mucho peso y peor que los peers → aquí debe ir la inversión de RRHH. Abajo-derecha: problema común del sector (pesa, pero no nos diferencia). Arriba-izquierda: vigilar. Abajo-izquierda: mantener/explotar.
- Punto hueco = diferencia frente a peers no significativa al 95%. **Solo cuenta como desventaja si es significativa**: priorizar una diferencia que puede ser azar es invertir en la palanca equivocada.
- *Marca y prestigio* no aparece en los cons (nadie se queja de la marca); su desventaja viene de que se **elogia menos** que en los peers.

In [ ]:
mat = comp[['cons_Deloitte', 'gap_balance', 'significativo']].copy()
mat['desventaja'] = -mat['gap_balance']
corte_x = mat['cons_Deloitte'].mean()
def cuadrante(r):
    # Solo es desventaja si es estadísticamente significativa: no se prioriza ruido
    if r.desventaja > 0 and r.significativo:
        return 'Prioridad alta' if r.cons_Deloitte >= corte_x else 'Vigilar'
    return 'Problema de sector' if r.cons_Deloitte >= corte_x else 'Mantener / explotar'
mat['cuadrante'] = mat.apply(cuadrante, axis=1)
mat = mat.sort_values(['cuadrante', 'desventaja'], ascending=[True, False])
RES['matriz'] = mat.round(3).to_dict(orient='index')
display(mat.round(2))

COL_CUAD = {'Prioridad alta': ROJO, 'Vigilar': '#D08A1E', 'Problema de sector': '#5F6B7A', 'Mantener / explotar': VERDE}
prio = mat[mat.cuadrante == 'Prioridad alta'].index.tolist()
if len(prio) == 1:
    titulo = f"Prioridad para RRHH: «{prio[0]}» — pesa en las quejas de Deloitte y estamos peor que los peers"
elif prio:
    titulo = f"Prioridad para RRHH: {' y '.join('«' + p + '»' for p in prio[:2])} — pesan en las quejas y estamos peor que los peers"
else:
    vig = mat[mat.cuadrante == 'Vigilar'].sort_values('desventaja', ascending=False).index[0]
    titulo = f"Ninguna queja grande de Deloitte es peor que en los peers; la mayor desventaja relativa está en «{vig}»"

fig, ax = figura(titulo, f'X: % de los cons de Deloitte. Y: desventaja del balance neto frente a la media de peers (pp). '
                 f'Corte vertical: peso medio ({corte_x:.1f}%). Hueco = no significativo.', figsize=(13, 7.5))
for c, r in mat.iterrows():
    col = COL_CUAD[r.cuadrante]
    ax.scatter(r.cons_Deloitte, r.desventaja, s=260, color=col if r.significativo else 'white', edgecolor=col, lw=2.2, zorder=3)
ax.axhline(0, color=TINTA2, lw=1); ax.axvline(corte_x, color=TINTA2, lw=1, ls='--')
ylim = max(abs(mat.desventaja).max() * 1.3, 3); ax.set_ylim(-ylim, ylim)
ax.set_xlim(0, mat.cons_Deloitte.max() * 1.25)
# Etiquetas sin solaparse: se estima su caja en unidades de datos y se separan verticalmente las que chocan
ancho_car = (ax.get_xlim()[1] / (13 * 0.85)) * 0.095
alto_txt = (2 * ylim / (7.5 * 0.75)) * 0.24
pos = {c: [r.cons_Deloitte + ancho_car * 1.5, r.desventaja + alto_txt * 0.6] for c, r in mat.iterrows()}
for _ in range(200):
    movido = False
    nombres = sorted(pos, key=lambda c: pos[c][1])
    for i, a in enumerate(nombres):
        for b in nombres[i + 1:]:
            solape_x = pos[a][0] < pos[b][0] + len(b) * ancho_car and pos[b][0] < pos[a][0] + len(a) * ancho_car
            if solape_x and abs(pos[a][1] - pos[b][1]) < alto_txt:
                pos[a][1] -= alto_txt * 0.1; pos[b][1] += alto_txt * 0.1; movido = True
    if not movido:
        break
for c, r in mat.iterrows():
    ax.annotate(c, (r.cons_Deloitte, r.desventaja), xytext=pos[c], textcoords='data', fontsize=11, color=TINTA, va='center',
                arrowprops=dict(arrowstyle='-', color='#BDC1C6', lw=0.8, shrinkA=0, shrinkB=9))
xr = ax.get_xlim()
for txt, xx, yy, ha in [('PRIORIDAD ALTA', xr[1], ylim * 0.92, 'right'), ('VIGILAR', xr[0] + 0.3, ylim * 0.92, 'left'),
                        ('PROBLEMA DE SECTOR (Deloitte igual o mejor)', xr[1], -ylim * 0.92, 'right'), ('MANTENER / EXPLOTAR', xr[0] + 0.3, -ylim * 0.92, 'left')]:
    ax.text(xx, yy, txt, ha=ha, va='center', fontsize=11, color=TINTA2, fontweight='bold', alpha=0.8)
ax.set_xlabel('Peso en los cons de Deloitte (%)'); ax.set_ylabel('Desventaja vs peers (pp)  ↑ peor')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
guardar(fig, '10_matriz_priorizacion')

**Lectura de la matriz.** **Ninguna de las grandes quejas de Deloitte es peor que en el mercado**:
- Carga de trabajo (20,6%), compensación (15,2%) y conciliación (12,8%) son **problemas del sector**. En las dos primeras Deloitte está incluso mejor que sus peers.
- La desventaja competitiva está en el cuadrante **"Vigilar"**: tres temas de **desarrollo profesional y orgullo de pertenencia** con desventaja significativa:
  - Aprendizaje y proyectos, −3,4 pp.
  - Carrera y promoción, −2,4 pp.
  - Marca y prestigio, −2,4 pp.
- Pesan poco hoy, pero *Aprendizaje y proyectos* **ha crecido desde 2020** (ver 7.1) y es el motivo de salida diferencial (ver 5). Por eso es la primera prioridad.

### 6.2 Recomendaciones para RRHH (máximo 3)

**1. Proyectos que hagan crecer: rotación y asignación transparente de proyectos.**
- *Por qué:* es la única categoría en la que Deloitte está peor que sus peers en elogios, en quejas y en motivos de salida. Además, se ha duplicado desde 2020 (del 4,4% al 7,2% de las quejas, mientras en los peers se mantiene en ~5%). Los empleados describen proyectos repetitivos, quedarse "atascado" en un mismo cliente y poca formación.
- *Qué hacer:*
  - Revisar con cada profesional, cada 12-18 meses, si quiere cambiar de proyecto o cliente.
  - Publicar internamente las oportunidades de proyecto.
  - Añadir a la encuesta de pulso la pregunta "¿estoy aprendiendo en mi proyecto actual?".
- *Cómo medirlo:* bajar el peso de esta queja del 6,7% al nivel de los peers (~4,8%) y que deje de destacar en las entrevistas de salida.

**2. Carrera clara para quien ya lleva años: criterios de promoción y evaluación transparentes.**
- *Por qué:* las quejas de promoción crecen con la antigüedad, del 1% de las quejas el primer año al 10,5% con más de 5 años. Están por encima de los peers (4,9% vs 3,7%) y apuntan como motivo de salida (+1,8 pp). Perder a los perfiles con más experiencia es lo más caro.
- *Qué hacer:*
  - Publicar los criterios de promoción por nivel.
  - Explicar cómo se calibran las evaluaciones.
  - Separar la conversación de carrera de la de evaluación anual.
- *Cómo medirlo:* peso de "Carrera y promoción" en las quejas de quienes llevan más de 3 años.

**3. Contar lo que ya hacemos mejor en la atracción de talento.**
- *Por qué:* Deloitte es la Big 4 mejor valorada (3,90) y la más recomendada (79%). Sus empleados elogian más el salario y los beneficios (+6,5 pp de balance neto) y se quejan menos de la carga de trabajo (+5,9 pp). En cambio, mencionan menos la marca como orgullo (−2,4 pp).
- *Qué hacer:* construir la campaña de captación sobre testimonios reales de empleados (compensación, beneficios y equipos) en lugar de apoyarse solo en el prestigio de la marca.
- *Vigilar:* desde 2020 la queja de carga de trabajo y la de conciliación han crecido en Deloitte (+2,5 y +2,7 pp) y no en los peers. La ventaja en carga de trabajo se está estrechando (−8,6 pp frente a los peers antes de 2020, −5,3 pp después) y conviene no prometer de más.

### 6.3 Limitaciones
- **Representatividad de Glassdoor:** escriben sobre todo los muy satisfechos y los muy descontentos, y no sabemos qué parte de la plantilla real representa cada reseña.
- **Mezcla de países y áreas:**
  - El dataset une una fuente centrada en Reino Unido y otra global (con mucho peso de India: aparecen términos como "GDS" o "India").
  - Mezcla auditoría, impuestos y consultoría.
  - Las diferencias entre firmas pueden deberse en parte a una distinta composición por país o área.
- **Precisión de los topics:**
  - Cada reseña recibe un solo tema aunque mencione varios ("long hours and low pay").
  - Un ~9-10% queda sin clasificar.
  - Algunos topics salen "de empresa" (ver 3.2).
  - La agrupación en categorías es una decisión humana, documentada y revisable.
- **Muestra:** 2.000 reseñas por empresa. Diferencias inferiores a ~2-3 pp no son significativas (por eso se marcan). Antes de 2020 solo hay unas 350 reseñas por empresa, así que la comparación temporal es orientativa.
- **Sentimiento:** el modelo es binario, solo en inglés y entrenado con críticas de cine. Falla con frases telegráficas (ver 2.1). Se usa solo como control de calidad.
- **Sesgo de selección en antiguos** y **cambio en el formulario de `recommend`** (más "sin opinión" con los años).
- **Correlación, no causalidad:** el análisis describe de qué se habla, no por qué se va la gente. Hay que contrastarlo con datos internos (entrevistas de salida, rotación, eNPS).

---
## 7. Extras
### 7.1 Evolución de las principales categorías: antes vs. desde 2020

In [ ]:
per = []
for periodo, g in df.groupby('periodo'):
    for grupo, firms in [('Deloitte', [TARGET]), ('Media peers', PEERS)]:
        pc = tabla_pct(g, 'cat_cons').loc[firms, CATS_SUST].mean()
        pp_ = tabla_pct(g, 'cat_pros').loc[firms, CATS_SUST].mean()
        for c in CATS_SUST:
            per.append({'periodo': periodo, 'grupo': grupo, 'categoria': c, 'cons': pc[c], 'pros': pp_[c]})
per = pd.DataFrame(per)
print('Reseñas por periodo (muestra):'); print(pd.crosstab(df.firm, df.periodo).reindex(FIRMS))
tab_per = per.pivot_table(index='categoria', columns=['grupo', 'periodo'], values='cons')
tab_per[('Deloitte', 'cambio')] = tab_per[('Deloitte', '2020 en adelante')] - tab_per[('Deloitte', 'Antes de 2020')]
tab_per[('Media peers', 'cambio')] = tab_per[('Media peers', '2020 en adelante')] - tab_per[('Media peers', 'Antes de 2020')]
tab_per = tab_per.sort_values(('Deloitte', '2020 en adelante'), ascending=False)
RES['periodo_cons'] = {f'{a}|{b}': v for (a, b), v in tab_per.round(3).to_dict().items()}
display(tab_per.round(1))

top6 = tab_per.index[:6]
sube = tab_per[('Deloitte', 'cambio')].idxmax(); baja = tab_per[('Deloitte', 'cambio')].idxmin()
fig, axes = figura(
    f"Desde 2020, en los cons de Deloitte gana peso «{sube}» ({tab_per.loc[sube, ('Deloitte', 'cambio')]:+.1f} pp) "
    f"y pierde «{baja}» ({tab_per.loc[baja, ('Deloitte', 'cambio')]:+.1f} pp)",
    '% de cons por categoría antes de 2020 (jul-2018 – dic-2019) y desde 2020. Seis categorías con más peso en Deloitte.',
    ncols=2, figsize=(15, 6), sharey=True)
y = np.arange(len(top6))[::-1]
for ax, grupo, col in [(axes[0], 'Deloitte', VERDE), (axes[1], 'Media peers', '#5F6B7A')]:
    a, b = tab_per.loc[top6, (grupo, 'Antes de 2020')], tab_per.loc[top6, (grupo, '2020 en adelante')]
    for yi, av, bv in zip(y, a, b):
        ax.annotate('', xy=(bv, yi), xytext=(av, yi), arrowprops=dict(arrowstyle='->', color=col, lw=2))
    ax.scatter(a, y, s=70, color='white', edgecolor=col, lw=2, zorder=3, label='Antes de 2020')
    ax.scatter(b, y, s=90, color=col, zorder=3, label='2020 en adelante')
    ax.set_title(grupo); ax.grid(axis='y', visible=False)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(5)); ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.legend(loc='lower right')
axes[0].set_yticks(y, top6)
guardar(fig, '11_evolucion_categorias')

**Lectura (7.1).** Desde 2020, en Deloitte ganan peso:
- **Aprendizaje y proyectos:** del 4,4% al 7,2% (+2,9 pp), frente a +0,6 en los peers.
- **Conciliación:** +2,7 pp, frente a +0,7.
- **Carga de trabajo:** +2,5 pp, frente a −0,6.

Pierden peso las quejas de **dirección** (−2,6 pp, igual que en los peers) y de **cultura** (−1,8 pp).

**Conclusión:** antes de 2020 Deloitte y sus peers estaban igualados en *Aprendizaje y proyectos* (4,4% vs 4,3%), así que **la desventaja actual es reciente**. Coincide con el trabajo en remoto o híbrido y con el crecimiento de la demanda de proyectos. La ventaja en carga de trabajo se mantiene, pero se ha estrechado. Antes de 2020 solo hay ~350 reseñas por empresa: hay que leerlo como tendencia, no como cifra exacta.

### 7.2 Por antigüedad (Deloitte)

In [ ]:
dD = df[(df.firm == TARGET) & (df.antiguedad != 'No indicada')]
ant_tab = (pd.crosstab(dD['antiguedad'], dD['cat_cons'], normalize='index') * 100).reindex(columns=CATS_SUST).fillna(0)
ant_tab = ant_tab.loc[[a for a in ORDEN_ANTIG if a in ant_tab.index and a != 'No indicada']]
n_ant = dD['antiguedad'].value_counts()
print('Reseñas de Deloitte por antigüedad:', n_ant.reindex(ant_tab.index).to_dict())
RES['antiguedad'] = ant_tab.round(2).to_dict(orient='index')
display(ant_tab.round(1))

cats_ord = ant_tab.mean().sort_values(ascending=False).index
primero, ultimo = ant_tab.iloc[0], ant_tab.iloc[-1]
crece = (ultimo - primero).idxmax()
fig, ax = figura(
    f"El primer año la queja principal es «{primero.idxmax()}» ({primero.max():.0f}%); con más de 5 años crece "
    f"«{crece}» (de {primero[crece]:.0f}% a {ultimo[crece]:.0f}%)",
    '% de los cons de Deloitte por categoría según antigüedad declarada (filas suman 100% con las categorías no mostradas).',
    figsize=(14, 4.8))
M = ant_tab[cats_ord].values
im = ax.imshow(M, cmap='Greens', aspect='auto', vmin=0)
ax.set_xticks(range(len(cats_ord)), [textwrap.fill(c, 14) for c in cats_ord], fontsize=10)
ax.set_yticks(range(len(ant_tab)), [f'{a}  (n={n_ant[a]})' for a in ant_tab.index])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f'{M[i, j]:.0f}%', ha='center', va='center', fontsize=10.5, color='white' if M[i, j] > M.max() * 0.6 else TINTA)
ax.grid(False); ax.spines[:].set_visible(False)
guardar(fig, '12_antiguedad')

**Lectura (7.2).** Las quejas cambian con la etapa profesional:
- **Primer año:** domina la **carga de trabajo** (23,5%). Es el choque de entrada.
- **Entre 1 y 5 años:** gana peso la **compensación** (17-18%) y, de 3 a 5 años, la **conciliación** (15,2%).
- **Más de 5 años:** sube con fuerza **Carrera y promoción** (del 0,9% al 10,5%) y **Dirección** (7,7%).

Esto orienta cada palanca a un colectivo: *onboarding* y gestión de picos de trabajo para los nuevos, revisión salarial en los años 1-3 y planes de carrera para los perfiles con experiencia (recomendación 2). Las reseñas sin antigüedad declarada se excluyen (n por tramo entre 209 y 462).

### 7.3 ¿Qué quejas separan a quien recomienda de quien no?
Se usa la muestra de las cuatro empresas para tener volumen. El **ratio** compara el peso de cada categoría en los cons de quienes **no** recomiendan frente a quienes **sí**: un ratio > 1 indica un tema que "rompe" la recomendación.

In [ ]:
rec = df[df.recomienda.isin(['Sí', 'No'])]
rt = (pd.crosstab(rec['recomienda'], rec['cat_cons'], normalize='index') * 100).reindex(columns=CATS_SUST).fillna(0).T
rt['ratio_No_vs_Si'] = rt['No'] / rt['Sí'].replace(0, np.nan)
recD = rec[rec.firm == TARGET]
rtD = (pd.crosstab(recD['recomienda'], recD['cat_cons'], normalize='index') * 100).reindex(columns=CATS_SUST).fillna(0).T
rt['ratio_Deloitte'] = rtD['No'] / rtD['Sí'].replace(0, np.nan)
rt = rt.dropna(subset=['ratio_No_vs_Si']).sort_values('ratio_No_vs_Si', ascending=False)   # fuera categorías sin cons
print('Reseñas con opinión:', rec.recomienda.value_counts().to_dict())
RES['recomendacion'] = rt.round(3).to_dict(orient='index')
display(rt.round(2))

fig, ax = figura(
    f"«{rt.index[0]}» es la queja que más distingue a quien NO recomienda: pesa {rt.iloc[0]['ratio_No_vs_Si']:.1f}× más "
    f"en sus cons que en los de quien sí recomienda",
    'Ratio entre el % de cons de cada categoría en reseñas que no recomiendan y en las que sí (4 empresas). Rombo: solo Deloitte.',
    figsize=(12.5, 6))
orden = rt.index[::-1]
vals = rt.loc[orden, 'ratio_No_vs_Si']
ax.barh(orden, vals, color=[ROJO if v > 1.2 else '#C9CED3' for v in vals], height=0.6)
ax.scatter(rt.loc[orden, 'ratio_Deloitte'], np.arange(len(orden)), marker='D', s=70, color=VERDE, zorder=3, label='Deloitte')
ax.axvline(1, color=TINTA2, lw=1, ls='--')
for i, v in enumerate(vals):
    ax.text(0.03, i, f'{v:.1f}×', va='center', fontsize=10.5, color='white' if v > 1.2 else TINTA)
ax.set_xlabel('Ratio (No recomienda / Sí recomienda)'); ax.legend(loc='lower right'); ax.grid(axis='y', visible=False)
guardar(fig, '13_recomendacion')

**Lectura (7.3).** Lo que separa a quien **no** recomienda de quien sí recomienda:
- **Dirección y management:** pesa **2,9 veces más** en sus quejas (3,0× en Deloitte).
- **Cultura y compañeros:** pesa **2,3 veces más** (2,1× en Deloitte).
- **Carga de trabajo:** pesa *menos* entre quienes no recomiendan (0,6×). Se queja de ella incluso quien recomienda.

Coincide con el contraste de rating de 4.1. Para la recomendación (y, por tanto, para la marca empleadora), **la calidad de los jefes y del ambiente pesa más que las horas**. Deloitte está en línea con sus peers en ambos temas, y mantenerlo es condición necesaria para todo lo demás.

In [ ]:
# Guardado de resultados clave (alimentan la presentación) y de la muestra etiquetada
def _json(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (pd.Period, pd.Timestamp)): return str(o)
    return str(o)
with open('resultados/metricas.json', 'w', encoding='utf-8') as fh:
    json.dump(RES, fh, ensure_ascii=False, indent=1, default=_json)
TOPICS['pros'].to_csv('resultados/topics_pros.csv', index=False)
TOPICS['cons'].to_csv('resultados/topics_cons.csv', index=False)
df.drop(columns=[c for c in df.columns if c.startswith('conf_')]).to_parquet('resultados/muestra_etiquetada.parquet')
print('Guardado: resultados/metricas.json, topics_*.csv, muestra_etiquetada.parquet; figuras en', FIG_DIR)

---
## Conclusiones de negocio

**Punto de partida.** Deloitte es la Big 4 mejor valorada en Glassdoor (rating **3,90**, **79%** de recomendación) y su ventaja se ha ampliado desde 2022 (4,00 frente a 3,85 de los peers). No hay una crisis que apagar: el reto es **defender el liderazgo** y corregir las debilidades antes de que crezcan.

**Puntos fuertes (para *employer branding* y selección)**
- **Compensación y beneficios:** es la mayor ventaja frente a los peers (+6,5 pp de balance neto). Hay más elogios y menos quejas que en la competencia.
- **Carga de trabajo más llevadera que en el sector:** sigue siendo la primera queja, pero pesa 5,9 pp menos que en los peers y empuja menos a irse (−6,4 pp en los antiguos).
- **Cultura y compañeros:** es lo más elogiado en todas las Big 4 (25% de los pros) y Deloitte está a la altura.
- **Empleados actuales especialmente satisfechos:** 3,99 de rating y 82% de recomendación.

**Áreas de mejora (donde estamos peor que el mercado)**
- **Aprendizaje y proyectos:** −3,4 pp. Es el motivo de salida diferencial (+2,7 pp en antiguos) y se ha duplicado desde 2020. **Prioridad 1.**
- **Carrera y promoción:** más quejas que los peers (4,9% vs 3,7%), concentradas en quienes llevan más de 5 años. **Prioridad 2.**
- **Marca y prestigio:** los empleados la citan menos como orgullo que en los peers (−2,4 pp).
- **A vigilar:** las quejas de carga de trabajo y de conciliación crecen desde 2020 en Deloitte y no en los peers.

**Palancas de retención**
1. Rotación y asignación transparente de proyectos, con un seguimiento periódico de "¿estoy aprendiendo?".
2. Criterios de promoción y evaluación claros, sobre todo a partir del tercer año.
3. Mantener la calidad del management y la cultura: no es una debilidad, pero es lo que más destruye la recomendación cuando falla (2,9× y 2,3×).
4. Entrevistas de salida y red de *alumni*: los antiguos son el 30% de las reseñas.

**Palancas de atracción**
1. Mensajes basados en lo que los empleados ya valoran más que en la competencia: compensación, beneficios y equipos.
2. Usar el liderazgo en rating y recomendación como prueba social, con seguimiento trimestral (PwC está muy cerca).
3. No apoyarse solo en el prestigio de la marca: los empleados de Deloitte lo mencionan menos que los de sus peers.

**Lo que no salió como se esperaba (dicho con transparencia)**
- El sentimiento de los cons coincide con lo esperado en "solo" el 81% de los casos, por una limitación del modelo con frases telegráficas (ver 2.1).
- Algunos topics capturan el nombre de la empresa pese a filtrarlo.
- Antes de 2020 hay pocas reseñas por empresa.

Ninguna de estas cuestiones cambia las conclusiones principales, que se apoyan en diferencias estadísticamente significativas.